# Lab 4: Run and inspect SmolVLA

**Goal:** Run a published vision-language-action policy, inspect its inputs and action chunk, and test an instruction while the simulator goal stays fixed. You will use the same LIBERO bowl-plate task and development states from the ACT labs.

**Inputs:** Pinned public SmolVLA and SmolVLM files, the local training example, and saved development results. **Compute:** Local GPU for one model call. **Cloud cost:** US$0 on this prepared machine. Run this notebook from the repository root after Labs 0–3. Read the [lab guide](README.md) first.

<a id="exercise"></a>

## Lab 4 exercise: Change an instruction

Use bowl-plate development state 0 with the published model. Keep the model, initial state, seed, ten-action execution setting, and bowl-plate evaluation goal fixed.

1. Run the canonical prompt `put the bowl on the plate`.
2. Run the paraphrase `place the bowl on the plate`.
3. Run the conflicting prompt `put the plate on the bowl`. Both objects are in this scene.
4. Record the model prompt, evaluation task, success, step count, and clipped action count for each run.
5. State one limit of three attempts on one simulator state.

The bowl-plate success check applies to all three runs. This exercise checks prompt sensitivity. It does not test selection between two evaluation goals. Read the [solution guide](../../solutions/04-smolvla.md) after you save your results.


## 1. Keep the two SmolVLA sources separate

The published LIBERO model **B** has its own architecture and saved processors. Its training data are unknown. Lab 5 builds learner **A** from the pinned SmolVLM backbone and trains new robot weights **C1** and **C2**. B is a useful inference reference. B is not the parent checkpoint for C.

The published model takes two camera images, an eight-value state, and a text task. It predicts 50 future seven-value actions. Its saved setting executes one action before a new policy call. The course also tested ten actions on development states.

In [ ]:
from pathlib import Path
import json
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


WORKSPACE = find_root()
assert (WORKSPACE / 'configs' / 'eval' / 'splits.yaml').is_file(), 'The required course file is missing.'
DATA_ROOT = WORKSPACE / '.local-data'
MODEL_DIR = DATA_ROOT / 'models' / 'smolvla-libero'
VLM_DIR = DATA_ROOT / 'models' / 'smolvlm2-500m-video-instruct'
reference = yaml.safe_load((WORKSPACE / 'manifests' / 'smolvla_reference.yaml').read_text())
assert MODEL_DIR.joinpath('model.safetensors').is_file()
assert VLM_DIR.joinpath('model.safetensors').is_file()
print('Published B:', reference['published_reference']['repo'])
print('Learner A source:', reference['learner_start']['repo'])
print('Same source:', reference['published_reference']['repo'] == reference['learner_start']['repo'])

### Reading block: 20 minutes

Read the architecture and initialization sections of the [SmolVLA paper](https://arxiv.org/abs/2506.01844). Find the image/text backbone and the action expert. Write one prediction: if the backbone is frozen and the robot action expert starts with new weights, will one optimizer step be enough for reliable control? Lab 5 checks this with real rollouts. Keep the paper's results separate from this course's LIBERO results.

## 2. Load B with its own processor

The loader checks the pinned local files, loads the published policy weights, and restores the processor that came with B. It changes only the selected execution length. Here we use ten actions, the setting selected on development states.

In [ ]:
from vla_lab.policy.smolvla import load_published_smolvla
from vla_lab.training.act import _frozen_training_dataset
from torch.utils.data._utils.collate import default_collate

policy, preprocessor, postprocessor = load_published_smolvla(MODEL_DIR, VLM_DIR, n_action_steps=10)
assert policy.config.chunk_size == 50 and policy.config.n_action_steps == 10
training_data = _frozen_training_dataset('bowl_plate', WORKSPACE / 'configs', DATA_ROOT, chunk_size=50)
example = training_data[0]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, key in zip(axes, ('observation.images.image', 'observation.images.image2')):
    axis.imshow(example[key].permute(1, 2, 0).numpy())
    axis.set_title(key)
    axis.axis('off')
plt.show()
print('State shape:', tuple(example['observation.state'].shape))
print('Target action chunk:', tuple(example['action'].shape))
print('Instruction:', example['task'])

In [ ]:
batch = preprocessor(default_collate([example]))
policy.reset()
with torch.no_grad():
    normalized_action = policy.select_action(batch)
    physical_action = postprocessor(normalized_action)
action = np.asarray(physical_action, dtype=np.float32).reshape(-1)
assert action.shape == (7,) and np.isfinite(action).all()
print('One seven-value action:', np.round(action, 3))
print('Action values outside [-1, 1]:', int(np.count_nonzero((action < -1) | (action > 1))))

## 3. Inspect the saved live results

The five-state check kept B and the state IDs fixed. It changed the number of actions executed before the next policy call. The selected ten-action setting then ran all 20 development states. A new execution length changes the observation schedule. It does not change the checkpoint weights.

In [ ]:
RESULTS = WORKSPACE / 'artifacts' / 'reference' / 'smolvla'
short = {}
for name in ('bowl_plate_n1_seeded_dev_0_4', 'bowl_plate_n10_seeded_dev_0_4'):
    folder = RESULTS / name
    summary = json.loads((folder / 'summary.json').read_text())
    rows = [json.loads(line) for line in (folder / 'episodes.jsonl').read_text().splitlines()]
    assert [row['state_id'] for row in rows] == [0, 1, 2, 3, 4]
    assert [row['seed'] for row in rows] == [20260930 + state for state in range(5)]
    assert summary['base_seed'] == 20260930
    short[name] = rows
    print(name, summary['successes'], '/', summary['episodes'])
all_result = json.loads((RESULTS / 'bowl_plate_n10_seeded_dev_all' / 'summary.json').read_text())
assert all_result['state_ids'] == list(range(20))
assert all_result['base_seed'] == 20260930
assert all_result['successes'] == reference['published_reference']['selected_result']['successes']
print('Selected ten-action result:', all_result['successes'], '/', all_result['episodes'])
labels = [str(index) for index in range(5)]
fig, axis = plt.subplots(figsize=(8, 3))
axis.plot(labels, [int(row['success']) for row in short['bowl_plate_n1_seeded_dev_0_4']], 'o-', label='one action')
axis.plot(labels, [int(row['success']) for row in short['bowl_plate_n10_seeded_dev_0_4']], 's--', label='ten actions')
axis.set(xlabel='Development state ID', ylabel='Success (1 or 0)', ylim=(-0.1, 1.1))
axis.legend()
plt.show()

## 4. Change text, keep the success check visible

The evaluation task is `put the bowl on the plate`. The saved check used the canonical prompt, the paraphrase `place the bowl on the plate`, and the conflicting prompt `put the plate on the bowl`. The scene has a bowl and a plate. The model, state, reset seed, and ten-action setting stayed fixed. A prompt change did not change the simulator's success check. One state shows an outcome for these prompts; it does not prove why the robot moved. Use the [exercise](#exercise) to repeat the check in new result directories.


In [ ]:
from vla_lab.evaluation.smolvla import evaluate_published_reference
import inspect

assert 'model_prompt' in inspect.signature(evaluate_published_reference).parameters
prompt_study = reference['published_reference']['prompt_study']
rows = []
for label, key in (('canonical', 'canonical_result'), ('paraphrase', 'paraphrase_result'), ('conflict', 'conflicting_result')):
    folder = WORKSPACE / prompt_study[key]
    summary = json.loads((folder / 'summary.json').read_text())
    row = json.loads((folder / 'episodes.jsonl').read_text())
    assert summary['state_ids'] == [prompt_study['state_id']]
    assert row['seed'] == prompt_study['reset_seed']
    assert row['evaluation_task'] == prompt_study['evaluation_task']
    assert row['model_prompt'] == summary['model_prompt']
    rows.append((label, row))
for label, row in rows:
    print(label, '| prompt:', row['model_prompt'], '| goal:', row['evaluation_task'], '| success:', row['success'], '| steps:', row['steps'])
print('One state cannot show why the robot moved. Inspect motion before you claim a language effect.')


## Completion check

You are done when you can identify B's two camera inputs, state, instruction, chunk size, and selected action execution length. State why B is a separate reference from learner A→C1→C2. Report the denominator for each saved success count. Explain why a conflicting prompt cannot change the simulator's fixed success predicate. Use the [exercise](#exercise) and then the [solution guide](../../solutions/04-smolvla.md).